# ORBITGUARD Stage 4: Tree-based anomaly classification

This notebook trains Decision Tree and Random Forest baselines, an XGBoost baseline, and a compact tuned XGBoost model. It uses the already-processed Stage 3 datasets and their official split. Cross-validation and hyperparameter selection use training rows only; the test set is held out until final reporting.

No model is trained from the original CSV in this notebook. `anomaly` stays separate as the target, `train` is not a feature, and the original source data is not overwritten. The tuned XGBoost inference pipeline and supporting metadata are saved under `models/`.

## 1. The models in plain English

- **Decision Tree:** A tree asks a sequence of questions about feature values. Each answer sends a record down a branch until a leaf predicts normal or anomalous. It is an easy-to-understand baseline, but one tree can change a lot with small data changes.
- **Random Forest:** A forest trains many decision trees on different samples and feature subsets, then combines their votes. Averaging usually makes the result more stable than relying on one tree.
- **XGBoost:** XGBoost is an efficient implementation of gradient-boosted decision trees. Boosting adds trees one at a time. Each new tree is trained to reduce the errors (more precisely, the loss gradients) left by the existing ensemble. The final score combines the contributions of all trees.

**Why investigate XGBoost here?** The dataset is a modest-sized table of numeric, engineered telemetry features. Tree ensembles can model nonlinear rules and interactions without feature scaling, which makes XGBoost a practical candidate. This is a reason to evaluate it, not evidence that it is the best model; the holdout results below must support any comparison.

The anomaly class is the minority class. We therefore report average precision (a precision-recall summary) and anomaly recall/precision in addition to accuracy. We do not rebalance the data in this stage.

## 2. Load the Stage 3 processed split

The loader validates that the files contain only the documented 18 features plus the separate `anomaly` target, and loads the Stage 3 fitted imputer. It does not create a random split. The row counts and label distributions below confirm that the official split was preserved.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd
from sklearn.metrics import classification_report

current_path = Path.cwd().resolve()
if current_path.name.lower() == "notebooks":
    current_path = current_path.parent
PROJECT_ROOT = next(
    (candidate for candidate in (current_path, *current_path.parents)
     if (candidate / "src" / "train.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find ORBITGUARD (src/train.py). Open this notebook from the "
        "project or clone the repository in Google Colab."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import ENGINEERED_FEATURE_COLUMNS, TARGET_COLUMN
from src.train import train_and_evaluate

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
train_frame = pd.read_csv(PROCESSED_DATA_DIR / "train_processed.csv")
test_frame = pd.read_csv(PROCESSED_DATA_DIR / "test_processed.csv")

feature_names = list(ENGINEERED_FEATURE_COLUMNS)
X_train = train_frame.loc[:, feature_names]
y_train = train_frame[TARGET_COLUMN]
X_test = test_frame.loc[:, feature_names]
y_test = test_frame[TARGET_COLUMN]

assert "anomaly" not in X_train.columns and "train" not in X_train.columns
assert "anomaly" not in X_test.columns and "train" not in X_test.columns
assert list(X_train.columns) == list(X_test.columns) == feature_names
print(f"Training features: {X_train.shape}; training target: {y_train.shape}")
print(f"Testing features:  {X_test.shape}; testing target:  {y_test.shape}")
print("Training labels:", y_train.value_counts().sort_index().to_dict())
print("Testing labels: ", y_test.value_counts().sort_index().to_dict())
print("Input leakage check: anomaly and train are not feature columns.")

Training features: (1594, 18); training target: (1594,)
Testing features:  (529, 18); testing target:  (529,)
Training labels: {0: 1273, 1: 321}
Testing labels:  {0: 416, 1: 113}
Input leakage check: anomaly and train are not feature columns.


## 3. Train and cross-validate the three baselines

The Decision Tree baseline uses one default Gini tree as a simple reference. The Random Forest baseline uses 300 Gini trees and averages their decisions. The XGBoost baseline uses 100 trees, depth 3, learning rate 0.1, and full row/feature sampling. The reusable training module defines these configurations and evaluates each with stratified three-fold cross-validation on the training partition, preserving each fold's class mix. The test data is not used in cross-validation or baseline fitting.

## 4. Tune XGBoost using training-only cross-validation

A compact randomized search samples 12 parameter combinations and scores each with the same stratified three-fold training-only cross-validation. This keeps the search bounded while investigating every requested XGBoost parameter:

- `n_estimators`: number of boosting rounds (trees); more rounds can learn more but take longer and may overfit.
- `max_depth`: maximum depth of each tree; deeper trees can model more detailed interactions but are more complex.
- `learning_rate`: how much each new tree contributes; smaller values learn more cautiously and often need more trees.
- `subsample`: fraction of training rows sampled for each boosting round; below 1.0 adds randomness that can reduce overfitting.
- `colsample_bytree`: fraction of input features sampled for each tree; this also adds diversity and can reduce overfitting.

Average precision is the selection score because anomalous segments are less common. This is a selection rule, not a claim about final test performance.

In [2]:
training_result = train_and_evaluate(
    processed_dir=PROCESSED_DATA_DIR,
    artifact_dir=MODEL_DIR,
    random_state=42,
    cv_splits=3,
    tuning_iterations=12,
)

print("Training and training-only CV completed.")
print(f"XGBoost best training-CV average precision: "
      f"{training_result['best_cv_average_precision']:.4f}")
print("Best XGBoost parameters:")
print(json.dumps(training_result["best_params"], indent=2))

Training and training-only CV completed.
XGBoost best training-CV average precision: 0.9554
Best XGBoost parameters:
{
  "subsample": 1.0,
  "n_estimators": 200,
  "max_depth": 3,
  "learning_rate": 0.1,
  "colsample_bytree": 0.8
}


## 5. Compare model configurations and evaluate the untouched test split

The table reports mean training-CV average precision and holdout test metrics. Accuracy alone can hide poor detection of the minority anomaly class, so also inspect balanced accuracy, anomaly precision/recall/F1, average precision, and ROC-AUC. The official test set is used only now, after tuning is complete. Compare observed results rather than assuming XGBoost is best.

In [3]:
comparison = training_result["comparison"].copy()
display(comparison.round(4))

configuration_rows = []
for model_name, model in training_result["models"].items():
    params = model.get_params(deep=False)
    if model_name == "Decision Tree baseline":
        selected = {key: params[key] for key in ["criterion", "max_depth", "min_samples_split", "class_weight", "random_state"]}
    elif model_name == "Random Forest baseline":
        selected = {key: params[key] for key in ["n_estimators", "criterion", "max_depth", "class_weight", "random_state"]}
    else:
        selected = {key: params[key] for key in ["n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree", "random_state"]}
    configuration_rows.append({"model": model_name, "configuration": json.dumps(selected, sort_keys=True)})
print("\nCompared model configurations:")
display(pd.DataFrame(configuration_rows))

print("\nHoldout test classification reports (class 0 = Normal, class 1 = Anomalous):")
for model_name, model in training_result["models"].items():
    print(f"\n{model_name}")
    print(classification_report(y_test, model.predict(X_test), labels=[0, 1],
                                target_names=["Normal", "Anomalous"], zero_division=0))
    print("Confusion matrix (rows=actual, columns=predicted; order 0, 1):")
    display(training_result["confusion_matrices"][model_name])

,model,cv_average_precision_mean,cv_average_precision_std,test_accuracy,test_balanced_accuracy,test_precision_anomaly,test_recall_anomaly,test_f1_anomaly,test_average_precision,test_roc_auc
0,Decision Tree baseline,0.6959,0.0755,0.9490,0.9224,0.8839,0.8761,0.8800,0.8009,0.9224
1,Random Forest baseline,0.9590,0.0245,0.9679,0.9344,0.9706,0.8761,0.9209,0.9684,0.9886
2,XGBoost baseline,0.9480,0.0284,0.9641,0.9288,0.9608,0.8673,0.9116,0.9643,0.9873
3,Tuned XGBoost,0.9554,0.0238,0.9584,0.9252,0.9333,0.8673,0.8991,0.9695,0.9900



Compared model configurations:


,model,configuration
0,Decision Tree baseline,"{""class_weight"": null, ""criterion"": ""gini"", ""m..."
1,Random Forest baseline,"{""class_weight"": null, ""criterion"": ""gini"", ""m..."
2,XGBoost baseline,"{""colsample_bytree"": 1.0, ""learning_rate"": 0.1..."
3,Tuned XGBoost,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.1..."



Holdout test classification reports (class 0 = Normal, class 1 = Anomalous):

Decision Tree baseline
              precision    recall  f1-score   support

      Normal       0.97      0.97      0.97       416
   Anomalous       0.88      0.88      0.88       113

    accuracy                           0.95       529
   macro avg       0.93      0.92      0.92       529
weighted avg       0.95      0.95      0.95       529

Confusion matrix (rows=actual, columns=predicted; order 0, 1):


predicted,0,1
actual,,
0,403,13
1,14,99



Random Forest baseline
              precision    recall  f1-score   support

      Normal       0.97      0.99      0.98       416
   Anomalous       0.97      0.88      0.92       113

    accuracy                           0.97       529
   macro avg       0.97      0.93      0.95       529
weighted avg       0.97      0.97      0.97       529

Confusion matrix (rows=actual, columns=predicted; order 0, 1):


predicted,0,1
actual,,
0,413,3
1,14,99



XGBoost baseline
              precision    recall  f1-score   support

      Normal       0.96      0.99      0.98       416
   Anomalous       0.96      0.87      0.91       113

    accuracy                           0.96       529
   macro avg       0.96      0.93      0.94       529
weighted avg       0.96      0.96      0.96       529

Confusion matrix (rows=actual, columns=predicted; order 0, 1):


predicted,0,1
actual,,
0,412,4
1,15,98



Tuned XGBoost
              precision    recall  f1-score   support

      Normal       0.96      0.98      0.97       416
   Anomalous       0.93      0.87      0.90       113

    accuracy                           0.96       529
   macro avg       0.95      0.93      0.94       529
weighted avg       0.96      0.96      0.96       529

Confusion matrix (rows=actual, columns=predicted; order 0, 1):


predicted,0,1
actual,,
0,409,7
1,15,98


### How boosting works in this implementation

The tuned XGBoost classifier begins with an initial prediction and adds one decision tree at each boosting round. Training calculates how the current predictions differ from the labels, then fits the next tree to reduce that remaining loss. The learning rate scales each tree's contribution. Depth, row subsampling, and feature subsampling constrain how complicated or correlated the added trees can become. The RandomizedSearchCV loop evaluates candidate settings using training folds only; its refitted best estimator then learns from the full training split.

## 6. Confirm and inspect saved artifacts

The main artifact is a loadable scikit-learn pipeline containing Stage 3's fitted imputer and the tuned XGBoost classifier. Separate preprocessing and feature-name artifacts make the inference contract explicit. The metadata and comparison table preserve configurations, CV results, holdout metrics, class counts, and library versions. Loading a joblib artifact should be done only from a trusted source and compatible Python package versions.

In [4]:
print("Model training confirmation: Decision Tree, Random Forest, XGBoost baseline, "
      "and tuned XGBoost were trained.")
print("\nFinal model artifact details:")
for artifact_name, artifact_path in training_result["artifact_paths"].items():
    artifact_path = Path(artifact_path)
    print(f"- {artifact_name}: {artifact_path} ({artifact_path.stat().st_size:,} bytes)")

model_path = training_result["artifact_paths"]["model"]
loaded_pipeline = joblib.load(model_path)
loaded_predictions = loaded_pipeline.predict(X_test)
assert (loaded_predictions == training_result["models"]["Tuned XGBoost"].predict(X_test)).all()
print("\nSaved XGBoost pipeline reload check: PASSED; predictions match the trained model.")
print("Stage 4 complete. No SHAP analysis or dashboard was started.")

Model training confirmation: Decision Tree, Random Forest, XGBoost baseline, and tuned XGBoost were trained.

Final model artifact details:
- model: C:\Users\HP\OneDrive\Desktop\orbit gaurd\models\xgboost_model.joblib (41,017 bytes)
- preprocessing_pipeline: C:\Users\HP\OneDrive\Desktop\orbit gaurd\models\preprocessing_pipeline.joblib (761 bytes)
- feature_information: C:\Users\HP\OneDrive\Desktop\orbit gaurd\models\feature_names.json (574 bytes)
- comparison: C:\Users\HP\OneDrive\Desktop\orbit gaurd\models\model_comparison.csv (945 bytes)
- metadata: C:\Users\HP\OneDrive\Desktop\orbit gaurd\models\model_metadata.json (7,995 bytes)



Saved XGBoost pipeline reload check: PASSED; predictions match the trained model.


Stage 4 complete. No SHAP analysis or dashboard was started.
